In [ ]:
# This cell can be deleted in the end.
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(1, '../')

# 07 Basic coupling example 
With the `pyFBS` coupling of different substructers can be performed in relatively simple manner. In this example a numerical example is used to demonstrate a basic coupling example with a virtual point transformation at the interface.

In [ ]:
import pyFBS

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

#### Example Datasets
Load the required predefined datasets:

In [ ]:
pos_xlsx = pyFBS.example_lab_testbench["meas"]["xlsx_coupling"]

stl_dir_A = pyFBS.example_lab_testbench["STL"]["A"]
stl_dir_B = pyFBS.example_lab_testbench["STL"]["B"]
stl_dir_AB = pyFBS.example_lab_testbench["STL"]["AB"]

df_acc_A = pd.read_excel(pos_xlsx, sheet_name='Sensors_A')
df_chn_A = pd.read_excel(pos_xlsx, sheet_name='Channels_A')
df_imp_A = pd.read_excel(pos_xlsx, sheet_name='Impacts_A')

df_acc_B = pd.read_excel(pos_xlsx, sheet_name='Sensors_B')
df_chn_B = pd.read_excel(pos_xlsx, sheet_name='Channels_B')
df_imp_B = pd.read_excel(pos_xlsx, sheet_name='Impacts_B')

df_acc_AB = pd.read_excel(pos_xlsx, sheet_name='Sensors_AB')
df_chn_AB = pd.read_excel(pos_xlsx, sheet_name='Channels_AB')
df_imp_AB = pd.read_excel(pos_xlsx, sheet_name='Impacts_AB')

## 3D view
Open 3D viewer in the background. With the 3D viewer the subplot capabilities of [PyVista](https://docs.pyvista.org/index.html) can be used.

In [ ]:
view3D = pyFBS.view3D(show_origin = False, show_axes = False,shape =  (1,3),title = "Overview")

Add the STL file of substructure A to the 1-1 subplot and show the corresponding accelerometer, channels and impacts.

In [ ]:
view3D.plot.subplot(0,0)
view3D.plot.isometric_view()
view3D.plot.add_text("A structure", position='upper_left', font_size=10, color="k", font="times", name="A_structure")

view3D.add_stl(stl_dir_A,color = "#83afd2",name = "A");
view3D.show_acc(df_acc_A)
view3D.show_imp(df_imp_A)
view3D.show_chn(df_chn_A)

Add the STL file of substructure B to the 1-2 subplot and show the corresponding accelerometer, channels and impacts.

In [ ]:
view3D.plot.subplot(0,1)
view3D.plot.isometric_view()
view3D.plot.add_text("B structure", position='upper_left', font_size=10, color="k", font="times", name="B_structure")

view3D.add_stl(stl_dir_B,color = "#83afd2",name = "B");
view3D.show_acc(df_acc_B,overwrite = False)
view3D.show_imp(df_imp_B,overwrite = False)
view3D.show_chn(df_chn_B,overwrite = False)

Add STL file of the assembly AB to the 1-2 subplot and show the corresponding reference accelerometer, channels and impacts.

In [ ]:
view3D.plot.subplot(0,2)
view3D.plot.isometric_view()
view3D.plot.add_text("AB structure", position='upper_left', font_size=10, color="k", font="times", name="AB_structure");

view3D.add_stl(stl_dir_AB,color = "#83afd2",name = "AB");
view3D.show_acc(df_acc_AB,overwrite = False)
view3D.show_imp(df_imp_AB,overwrite = False)
view3D.show_chn(df_chn_AB,overwrite = False)

Each separate subplot view can also be linked or unlinked:

In [ ]:
view3D.plot.link_views()
#view3D.plot.unlink_views()

## Numerical model
Load the corresponding .full and .ress file from the example datasets. For more information on .full and .ress files refer to the *03_FRF_synthetization.ipynb* example

In [ ]:
full_file_AB = pyFBS.example_lab_testbench["FEM"]["AB_full"]
ress_file_AB = pyFBS.example_lab_testbench["FEM"]["AB_rst"]

full_file_B = pyFBS.example_lab_testbench["FEM"]["B_full"]
ress_file_B = pyFBS.example_lab_testbench["FEM"]["B_rst"]

full_file_A = pyFBS.example_lab_testbench["FEM"]["A_full"]
ress_file_A = pyFBS.example_lab_testbench["FEM"]["A_rst"]

Create an MK model for each component:

In [ ]:
MK_A = pyFBS.MK_model(ress_file_A,full_file_A,no_modes = 100,allow_pickle= True,recalculate = False)
MK_B = pyFBS.MK_model(ress_file_B,full_file_B,no_modes = 100,allow_pickle= True,recalculate = False)
MK_AB = pyFBS.MK_model(ress_file_AB,full_file_AB,no_modes = 100,allow_pickle= True,recalculate = False)

Update locations of channels and impacts to snap to the nearest FE node.

In [ ]:
df_chn_A_up = MK_A.update_locations_df(df_chn_A)
df_imp_A_up = MK_A.update_locations_df(df_imp_A)

df_chn_B_up = MK_B.update_locations_df(df_chn_B)
df_imp_B_up = MK_B.update_locations_df(df_imp_B)

df_chn_AB_up = MK_AB.update_locations_df(df_chn_AB)
df_imp_AB_up = MK_AB.update_locations_df(df_imp_AB)

Perform the FRF sythetization for each component based on the updated locations.

In [ ]:
MK_A.FRF_synth(df_chn_A_up,df_imp_A_up,f_start = 0,modal_damping = 0.003)
MK_B.FRF_synth(df_chn_B_up,df_imp_B_up,f_start = 0,modal_damping = 0.003)
MK_AB.FRF_synth(df_chn_AB_up,df_imp_AB_up,f_start = 0,modal_damping = 0.003)

## Virtual point transformation
The VPT can be performed directly on the generated data. See the *04_VPT.ipynb* example for more options and details.

In [ ]:
df_vp = pd.read_excel(pos_xlsx, sheet_name='VP_Channels')
df_vpref = pd.read_excel(pos_xlsx, sheet_name='VP_RefChannels')

vpt_A = pyFBS.VPT(df_chn_A_up,df_imp_A_up,df_vp,df_vpref)
vpt_B = pyFBS.VPT(df_chn_B_up,df_imp_B_up,df_vp,df_vpref)

Apply the defined VP transformation on the FRFs:

In [ ]:
vpt_A.apply_VPT(MK_A.freq,MK_A.FRF)
vpt_B.apply_VPT(MK_B.freq,MK_B.FRF)

Extract the requried FRFs and the frequency vector:

In [ ]:
freq = MK_A.freq
Y_A = vpt_A.vptData
Y_B = vpt_B.vptData

## LM-FBS Coupling
First the compatibility and the equiliubrium condition has to be defined through the signed Boolean matrices. For this example the 6 VP DoFs at the interface are coupled.

In [ ]:
Y_AnB = np.zeros((2000,12+18,12+18),dtype = complex)

Y_AnB[:,0:12,0:12] = Y_A
Y_AnB[:,12:,12:] =   Y_B

k = 6
Bu = np.zeros((k,12+18))
Bu[:k,6:6+k] = 1*np.eye(k)
Bu[:k,12:12+k] = -1*np.eye(k)

plt.figure()
plt.imshow(Bu)

Bf = np.zeros((k,12+18))
Bf[:k,6:6+k] = 1*np.eye(k)
Bf[:k,12:12+k] = -1*np.eye(k)

plt.figure()
plt.imshow(Bf)

Apply the LM-FBS based on the defined coompatibility and equilibrium conditions.

In [ ]:
Y_ABn = np.zeros_like(Y_AnB,dtype = complex)

Y_int = Bu@Y_AnB@Bf.T
Y_ABn = Y_AnB - Y_AnB@Bf.T@np.linalg.pinv(Y_int)@Bu@Y_AnB

#### Final results
First extract the FRFs at the reference DoFs:

In [ ]:
arr_coup = [0,1,2,3,4,5,18,19,20,21,22,23,24,25,26,27,28,29]
Y_AB_coupled = Y_ABn[:,arr_coup,:][:,:,arr_coup]
Y_AB_ref = MK_AB.FRF

The coupled and the reference results can then be compared:

In [ ]:
s1 = 0
s2 = 6

display(df_chn_AB_up.loc[[s1]])
display(df_imp_AB_up.loc[[s2]])

plt.figure(figsize = (10,6))
plt.subplot(211)
plt.semilogy(freq,np.abs(Y_AB_ref[:,s1,s2]))
plt.semilogy(freq,np.abs(Y_AB_coupled[:,s1,s2]))

plt.xlim(0,2000)

plt.subplot(413)
plt.plot(freq,np.angle(Y_AB_ref[:,s1,s2]))
plt.plot(freq,np.angle(Y_AB_coupled[:,s1,s2]))


plt.xlim(0,2000)

## Result animation ODS
The coupling results can be animated directly on accelerometers. First open a 3D display:

In [ ]:
view3D_an = pyFBS.view3D(show_origin = False, show_axes = False,title = "Animation")

Load the example datasets and display accelerometer, channels and impacts:

In [ ]:
stl_dir = pyFBS.example_lab_testbench["STL"]["AB"]
view3D_an.add_stl(stl_dir,color = "#83afd2",name = "AB")

view3D_an.show_acc(df_acc_AB,overwrite = False)
view3D_an.show_imp(df_imp_AB_up,overwrite = False)
view3D_an.show_chn(df_chn_AB_up,overwrite = False)

view3D_an.label_acc(df_acc_AB,name = "acc_AB")
view3D_an.label_chn(df_chn_AB_up,name = "chn_AB")
view3D_an.label_imp(df_imp_AB_up,name = "imp_AB")

Select the input location and the frequency line for the ODS animation:

In [ ]:
freq_sel = 20
s1 = 15
select_in = 6

plt.figure(figsize = (10,6))
plt.subplot(211)

plt.semilogy(freq,np.abs(Y_AB_ref[:,s1,select_in]))
plt.semilogy(freq,np.abs(Y_AB_coupled[:,s1,select_in]))

plt.semilogy(freq[freq_sel],np.abs(Y_AB_ref[freq_sel,s1,select_in]),'o',color = "k")

plt.xlim(0,2000)

#### Coupled results
Accelerometer animation based on the coupled results:

In [ ]:
ann = pyFBS.orient_in_global(Y_AB_coupled[freq_sel,:,select_in],df_chn_AB_up,df_acc_AB)

mode_dict = pyFBS.dict_animation(ann,"object",object_list = view3D_an.global_acc,r_scale=30)
mode_dict["freq"] = freq[freq_sel]
view3D_an.add_objects_animation(mode_dict,run_animation = True,add_note= True)

#### Reference
Also the reference FRFs can be animated and compared with the final coupled results:

In [ ]:
ann = pyFBS.orient_in_global(Y_AB_ref[freq_sel,:,select_in],df_chn_AB_up,df_acc_AB)

mode_dict = pyFBS.dict_animation(ann,"object",object_list = view3D_an.global_acc,r_scale=30)
mode_dict["freq"] = freq[freq_sel]
view3D_an.add_objects_animation(mode_dict,run_animation = True,add_note= True)